In [12]:
!gdown --folder 1scEa2hx2wgxOW2h40Hy7k1PjeF4NCa3j

Retrieving folder contents
Processing file 1lzLTLb11vtZv93z3LHRyKTRBBCOuQ5kB customers.csv
Processing file 1TqVJyvdS49LqbYViawTS3Ibybe8cO1ea geography.csv
Processing file 1Se8smRg3Z8z9CeJpoTgnSgUO8mZY6nwy inventory.csv
Processing file 1cPUVjWzwIPyt8DcHTw3XifzivsMpGmiE order_items.csv
Processing file 1Ts0eZ7X7GJMFD54VAseqKCLY2RzuVIY_ orders.csv
Processing file 1QQ3qosngd6cL3g9VOuJmK2NGLNKyMuJn payments.csv
Processing file 1bN_Pl274N0Xb2j97X9KIUb9vVkz4ZTBu products.csv
Processing file 10_Dg521vWpUiJazGAgLZUy_k4i1zZrOM promotions.csv
Processing file 1acUhr1IR0oN-3pJcrfKEHTSAMt8PgoKY returns.csv
Processing file 1hrI4kJMhVWyT48YlnW793OB2jSzy-d3F reviews.csv
Processing file 1nYMjy1D0nePsvQ71PMzoa7Q-pGDPkm94 sales.csv
Processing file 1BECp2dQXTpOG05ex4eKB3eQiDubbwdGj sample_submission.csv
Processing file 1l4Hog9E3D2IDx5J2LfXwl81v7sFvXtv1 shipments.csv
Processing file 1EM-ufHHisbaNn46NWBAtatz6vyaoF5H4 web_traffic.csv
Retrieving folder contents completed
Building directory structure
Building di

In [13]:
import pandas as pd
import numpy as np

In [14]:
import pandas as pd

products = pd.read_csv('/content/raw_datasets/products.csv')
customers = pd.read_csv('/content/raw_datasets/customers.csv')
promotions = pd.read_csv('/content/raw_datasets/promotions.csv')
geography = pd.read_csv('/content/raw_datasets/geography.csv')

orders = pd.read_csv('/content/raw_datasets/orders.csv')
order_items = pd.read_csv('/content/raw_datasets/order_items.csv')
payments = pd.read_csv('/content/raw_datasets/payments.csv')
shipments = pd.read_csv('/content/raw_datasets/shipments.csv')
returns = pd.read_csv('/content/raw_datasets/returns.csv')
reviews = pd.read_csv('/content/raw_datasets/reviews.csv')

sales = pd.read_csv('/content/raw_datasets/sales.csv')

inventory = pd.read_csv('/content/raw_datasets/inventory.csv')
web_traffic = pd.read_csv('/content/raw_datasets/web_traffic.csv')

/tmp/ipykernel_14951/2279912930.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('/content/raw_datasets/order_items.csv')


# **Question 1:**

In [15]:

orders["order_date"] = pd.to_datetime(orders["order_date"])

orders = orders.sort_values(["customer_id", "order_date"])

orders["prev_date"] = orders.groupby("customer_id")["order_date"].shift(1)
orders["gap"] = (orders["order_date"] - orders["prev_date"]).dt.days

gaps = orders.dropna(subset=["gap"])

median_gap = gaps["gap"].median()
print(median_gap)

144.0


In [17]:
import pandas as pd


# Convert datetime
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Tính inter-order gap và median
median_gap = (
    orders
    .sort_values(['customer_id', 'order_date'])  # sắp xếp theo khách + thời gian
    .assign(prev_date=lambda df: df.groupby('customer_id')['order_date'].shift(1))  # đơn trước
    .assign(gap=lambda df: (df['order_date'] - df['prev_date']).dt.days)  # khoảng cách ngày
    .loc[lambda df: df['gap'].notna(), 'gap']  # bỏ khách chỉ có 1 đơn
    .median()  # lấy trung vị
)

print("Median inter-order gap (days):", median_gap)

Median inter-order gap (days): 144.0


# Question 2:

In [19]:

# Tính margin cho từng sản phẩm
# margin = (price - cogs) / price
products["margin"] = (products["price"] - products["cogs"]) / products["price"]

# Group theo segment và lấy trung bình margin
segment_margin = products.groupby("segment")["margin"].mean()

# Sắp xếp giảm dần để tìm segment cao nhất
segment_margin = segment_margin.sort_values(ascending=False)

# In kết quả
print("Q2 - Margin trung bình theo segment:")
print(segment_margin)

# Lấy segment cao nhất
print("=> Đáp án:", segment_margin.idxmax())

Q2 - Margin trung bình theo segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64
=> Đáp án: Standard


# Question 3:

In [ ]:
returns = pd.read_csv("returns.csv")
# Join returns với products để lấy category
df = returns.merge(products, on="product_id", how="inner")

# Lọc chỉ lấy sản phẩm thuộc category Streetwear
streetwear_returns = df[df["category"] == "Streetwear"]

# Đếm số lần xuất hiện của từng lý do trả hàng
reason_counts = streetwear_returns["return_reason"].value_counts()

print("Q3 - Return reason (Streetwear):")
print(reason_counts)

# Lấy lý do phổ biến nhất
print("=> Đáp án:", reason_counts.idxmax())

Q3 - Return reason (Streetwear):
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
=> Đáp án: wrong_size


# **Question 4:**

In [22]:
# Group theo traffic_source và tính trung bình bounce_rate
bounce_avg = web_traffic.groupby("traffic_source")["bounce_rate"].mean()

# Sắp xếp tăng dần để tìm thấp nhất
bounce_avg = bounce_avg.sort_values()

print("Q4 - Bounce rate trung bình:")
print(bounce_avg)

# Lấy source có bounce_rate thấp nhất
print("=> Đáp án:", bounce_avg.idxmin())

Q4 - Bounce rate trung bình:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
=> Đáp án: email_campaign


# Question 5:

In [23]:

# Kiểm tra promo_id có null hay không
# notna() → True nếu có promo

promo_ratio = order_items["promo_id"].notna().mean()

# Convert sang %
promo_percent = promo_ratio * 100

print("Q5 - % có promo:", promo_percent)

Q5 - % có promo: 38.663493169565214


# Question 6:

In [24]:
# Join orders với customers để lấy age_group
df = orders.merge(customers, on="customer_id", how="inner")

# Lọc bỏ những khách không có age_group
df = df[df["age_group"].notna()]

# Đếm số đơn mỗi khách trong từng nhóm tuổi
orders_per_customer = df.groupby(["age_group", "customer_id"]) \
                        .size() \
                        .reset_index(name="order_count")

# Tính trung bình số đơn trên mỗi khách theo nhóm tuổi
avg_orders = orders_per_customer.groupby("age_group")["order_count"].mean()

# Sắp xếp giảm dần
avg_orders = avg_orders.sort_values(ascending=False)

print("Q6 - Avg orders per customer:")
print(avg_orders)

# Lấy nhóm cao nhất
print("=> Đáp án:", avg_orders.idxmax())

Q6 - Avg orders per customer:
age_group
55+      7.268731
45-54    7.220264
35-44    7.206159
25-34    7.112230
18-24    7.068577
Name: order_count, dtype: float64
=> Đáp án: 55+


# Question 7:

In [25]:
# Join orders với geography để lấy region
df = orders.merge(geography, on="zip", how="left")

# Join thêm order_items để tính revenue
df = df.merge(order_items, on="order_id", how="left")

# Tính revenue từng dòng item
# revenue = quantity * unit_price - discount
df["revenue"] = df["quantity"] * df["unit_price"] - df["discount_amount"]

# Group theo region và tính tổng revenue
region_revenue = df.groupby("region")["revenue"].sum()

# Sắp xếp giảm dần
region_revenue = region_revenue.sort_values(ascending=False)

print("Q7 - Revenue theo region:")
print(region_revenue)

# Lấy region cao nhất
print("=> Đáp án:", region_revenue.idxmax())

Q7 - Revenue theo region:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: revenue, dtype: float64
=> Đáp án: East


# Question 8:

In [26]:
# Lọc đơn hàng bị cancel
cancelled_orders = orders[orders["order_status"] == "cancelled"]

# Join với payments (1-1)
df = cancelled_orders.merge(payments, on="order_id", how="left")

# Đếm số lần mỗi payment method
method_counts = df["payment_method_y"].value_counts()

print("Q8 - Payment method (cancelled):")
print(method_counts)

# Lấy phổ biến nhất
print("=> Đáp án:", method_counts.idxmax())

Q8 - Payment method (cancelled):
payment_method_y
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
=> Đáp án: credit_card


# Question 9:

In [27]:
# --- Tử số: số return theo size ---
returns_df = returns.merge(products, on="product_id", how="left")

returns_count = returns_df.groupby("size").size()

# --- Mẫu số: số order_items theo size ---
items_df = order_items.merge(products, on="product_id", how="left")

total_items = items_df.groupby("size").size()

# --- Return rate ---
return_rate = (returns_count / total_items)

# Sắp xếp giảm dần
return_rate = return_rate.sort_values(ascending=False)

print("Q9 - Return rate theo size:")
print(return_rate)

# Lấy size cao nhất
print("=> Đáp án:", return_rate.idxmax())

Q9 - Return rate theo size:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64
=> Đáp án: S


# Question 10:

In [28]:
# Group theo số kỳ trả góp
installment_avg = payments.groupby("installments")["payment_value"].mean()

# Sắp xếp giảm dần
installment_avg = installment_avg.sort_values(ascending=False)

print("Q10 - Avg payment theo installments:")
print(installment_avg)

# Lấy cao nhất
print("=> Đáp án:", installment_avg.idxmax())

Q10 - Avg payment theo installments:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
=> Đáp án: 6
